In [1]:
import scanpy as sc
import pandas as pd
import numpy as np

/opt/conda/envs/scenv/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/opt/conda/envs/scenv/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/opt/conda/envs/scenv/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
adata_ = sc.read_h5ad("../adata_all_harm_ribo.h5ad")
adata_ = adata_[adata_.obs.cell_type_final != "Unlabeled"]
adata_

View of AnnData object with n_obs × n_vars = 1440640 × 43087
    obs: 'GSE_id', 'GSM_id', 'Age', 'Tissue_global', 'cell_type_final', 'Age_bin'
    uns: 'Tissue_global_colors'
    obsm: 'X_scANVI', 'X_scVI', 'X_umap', '_scvi_extra_categorical_covs'

In [3]:
unif_cts = {
    "Ventricular Cardiomyocyte": "Ventricular cardiomyocytes",
    "Atrial Cardiomyocyte": "Atrial cardiomyocytes",
    "Bronchial smooth muscle cell": "Bronchial SMCs",
    "CD4T": "CD4 T cells",
    "CD8T": "CD8 T cells",
    "Capillary endothelial cell": "Capillary endothelium",
    "DC": "Dendritic cells",
    "Endothelial cell of lymphatic vessel": "Lymphatic endothelium",
    "LE": "Lymphatic endothelium",
    "Lymphatic Endothelial cell": "Lymphatic endothelium",
    "Macrophage": "Monocytes/macrophages",
    "Mono+mac": "Monocytes/macrophages",
    "Monocyte": "Monocytes/macrophages",
    "NK": "NK cells",
    "Respiratory tract goblet cell": "Goblet cells",
    "VE": "Vein endothelium",
    "Vein endothelial cell": "Vein endothelium",
    "Endothelial cell of artery": "Artery endothelium",
    "gdT": "γδ T cells",
    "Pulmonary alveolar type 1 cell": "T1 pneumocytes",
    "Pulmonary alveolar type 2 cell": "T2 pneumocytes",
    "B cell": "B cells",
    "Crypt stem cell": "Crypt stem cells",
    "Enterocyte": "Enterocytes",
    "Fibroblast": "Fibroblasts",
    "Mast cell": "Mast cells",
    "Transit amplifying cell": "Mid-crypt cells",
    "Endothelial cell": "Endothelial cells",
    "Mural cell": "Mural cells",
    "Adipocyte": "Adipocytes",
    "Neural cell": "Neural cells",
    "Goblet cell": "Goblet cells",
    "Paneth cell": "Paneth cells",
    "Plasma cell": "Plasma cells",
    "Epithelial cell": "Epithelial cells",
    "Adventitial cell": "Adventitial cells",
    "Basal cell": "Basal cells",
    "Club cell": "Club cells",
    "Multiciliated epithelial cell": "Multiciliated epithelial cells",
    "Melanocyte": "Melanocytes",
    "Pericyte": "Pericytes",
    "Treg": "Tregs",
    "Differentiated_KC": "Diff. keratinocytes",
    "Undifferentiated_KC": "Undiff. keratinocytes",
}

adata_.obs["cell_type"] = adata_.obs["cell_type_final"].replace(unif_cts)

/tmp/ipykernel_84505/2623552243.py:48: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata_.obs["cell_type"] = adata_.obs["cell_type_final"].replace(unif_cts)
/tmp/ipykernel_84505/2623552243.py:48: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_.obs["cell_type"] = adata_.obs["cell_type_final"].replace(unif_cts)


In [4]:
for ct in adata_.obs.cell_type_final.unique():
    if ct not in unif_cts.keys():
        print(ct)

Schwann cell
Proliferating_KC
LC
NKT
Basophil
Alveolar adventitial fibroblast
Vascular associated smooth muscle cell
Neutrophil
Podocyte
MyoFib+VSMC
Tuft cell
Enteroglial cell
Type L enteroendocrine cell
Enteroendocrine cell
Myofibroblast
Mesothelial cell


In [5]:
pd.set_option("display.max_rows", None)


adata_.obs[["Tissue_global", "cell_type"]].value_counts().rename(
    "count"
).reset_index().sort_values(
    ["Tissue_global", "count"], ascending=[True, False]
).set_index(["Tissue_global", "cell_type"])["count"]


Tissue_global  cell_type                             
Colon          Plasma cells                               21318
               Crypt stem cells                           16782
               Enterocytes                                16669
               Goblet cells                                8014
               CD4 T cells                                 7582
               Fibroblasts                                 7483
               CD8 T cells                                 5808
               B cells                                     4663
               Mid-crypt cells                             2480
               γδ T cells                                  2368
               Endothelial cells                           2026
               Monocytes/macrophages                       1617
               Tuft cell                                   1553
               Mast cells                                  1194
               Enteroglial cell                   

In [6]:
# число экспрессируемых генов в каждой клетке
n_expr_genes_per_cell = adata_.X.getnnz(axis=1)

# фильтруем клетки, где экспрессируется >= 500 генов
cells_to_keep = n_expr_genes_per_cell >= 500

# оставляем только эти клетки
adata_ = adata_[cells_to_keep].copy()

In [7]:
(n_expr_genes_per_cell < 500).sum()

np.int64(80564)

In [8]:
# 2. Создаем копию метаданных для вычисления фильтров
df = adata_.obs[["Tissue_global", "cell_type", "GSM_id"]].copy()

# Шаг А: Считаем количество клеток для каждой комбинации (Ткань + Кл_тип + GSM_id)
# Фильтруем, оставляя только те группы, где клеток >= 100
counts_cells = (
    df.groupby(["Tissue_global", "cell_type", "GSM_id"], observed=True)
    .size()
    .reset_index(name="cell_count")
)
valid_cells_groups = counts_cells[counts_cells["cell_count"] >= 100]

# Шаг Б: Считаем количество уникальных GSM_id для каждой комбинации (Ткань + Кл_тип)
counts_gsm = (
    valid_cells_groups.groupby(["Tissue_global", "cell_type"], observed=True)["GSM_id"]
    .nunique()
    .reset_index(name="gsm_count")
)
valid_gsm_groups = counts_gsm[counts_gsm["gsm_count"] >= 3]

# Шаг В: Объединяем фильтры, чтобы получить финальный список валидных комбинаций
final_valid_combinations = valid_cells_groups.merge(
    valid_gsm_groups[["Tissue_global", "cell_type"]],
    on=["Tissue_global", "cell_type"],
    how="inner",
)

# Шаг Г: Безопасная фильтрация строк с сохранением оригинальных индексов клеток
df["triplet_key"] = (
    df["Tissue_global"].astype(str)
    + "_"
    + df["cell_type"].astype(str)
    + "_"
    + df["GSM_id"].astype(str)
)
final_valid_combinations["triplet_key"] = (
    final_valid_combinations["Tissue_global"].astype(str)
    + "_"
    + final_valid_combinations["cell_type"].astype(str)
    + "_"
    + final_valid_combinations["GSM_id"].astype(str)
)
# Фильтруем оригинальный df по ключу, сохраняя его истинные индексы клеток (баркоды)
keep_cells = df.index[df["triplet_key"].isin(final_valid_combinations["triplet_key"])]

# 3. Фильтрация самого объекта AnnData
adata = adata_[keep_cells].copy()

# Вывод промежуточного результата для проверки
print(f"Исходный размер: {adata_.n_obs} клеток")
print(f"Размер после фильтрации: {adata.n_obs} клеток")


Исходный размер: 1360076 клеток
Размер после фильтрации: 1297626 клеток


In [10]:
df.head()

,Tissue_global,cell_type,GSM_id,triplet_key
AAACCTGAGGCCGAAT-1-GSM3758115,Skin,Fibroblasts,GSM3758115,Skin_Fibroblasts_GSM3758115
AAACCTGAGTTAAGTG-1-GSM3758115,Skin,CD8 T cells,GSM3758115,Skin_CD8 T cells_GSM3758115
AAACCTGCAGTCAGAG-1-GSM3758115,Skin,Schwann cell,GSM3758115,Skin_Schwann cell_GSM3758115
AAACCTGGTCTTCTCG-1-GSM3758115,Skin,Vein endothelium,GSM3758115,Skin_Vein endothelium_GSM3758115
AAACCTGGTGCAACTT-1-GSM3758115,Skin,Dendritic cells,GSM3758115,Skin_Dendritic cells_GSM3758115


In [11]:
# 1. Считаем количество клеток для каждой пары
counts_df = (
    adata.obs[["Tissue_global", "cell_type"]]
    .value_counts()
    .rename("count")
    .reset_index()
)

# 2. Разворачиваем в матричную таблицу
raw_pivot = counts_df.pivot(index="cell_type", columns="Tissue_global", values="count")

# 3. Превращаем числа в целые, а вместо NaN (NA) вставляем незаметную точку "."
# Мы переводим в формат строк (str), так как точка — это текстовый символ
pivot_table = raw_pivot.fillna(".").astype(str)

# Опционально: убираем названия осей index и columns для более чистого вида таблицы
pivot_table.index.name = None
pivot_table.columns.name = None

pivot_table

,Colon,Heart_Cell,Heart_Nuclei,Ileum,Kidney,Lung,Skin
Adipocytes,.,.,5136.0,.,.,.,.
Adventitial cells,.,.,.,.,.,4025.0,.
Alveolar adventitial fibroblast,.,.,.,.,.,5755.0,.
Atrial cardiomyocytes,.,.,38152.0,.,.,.,.
B cells,4303.0,.,.,7527.0,.,1007.0,.
Basal cells,.,.,.,.,.,1660.0,.
Basophil,.,.,.,.,.,847.0,.
Bronchial SMCs,.,.,.,.,.,1141.0,.
CD4 T cells,7300.0,1483.0,2790.0,6810.0,1310.0,10474.0,11652.0
CD8 T cells,5311.0,1815.0,3674.0,7302.0,2962.0,12405.0,7263.0


In [12]:
# 3. Считаем, в скольких тканях (столбцах) представлена каждая клетка
# .notna().sum(axis=1) считает количество заполненных ячеек в каждой строке
tissue_counts_per_cell = raw_pivot.notna().sum(axis=1)

# 4. Разделяем типы клеток на две группы по индексам
unique_cells_idx = tissue_counts_per_cell[tissue_counts_per_cell == 1].index
shared_cells_idx = tissue_counts_per_cell[tissue_counts_per_cell >= 2].index

# 5. Выделяем две отдельные таблицы и оформляем их (заменяем NaN на ".")
# Таблица 1: Клеточные типы, которые есть ТОЛЬКО в одной ткани
uniq_t_table = raw_pivot.loc[unique_cells_idx].fillna(".").astype(str)
uniq_t_table.index.name = None
uniq_t_table.columns.name = None

# Таблица 2: Клеточные типы, которые есть в 2 и более тканях
shar_t_table = raw_pivot.loc[shared_cells_idx].fillna(".").astype(str)
shar_t_table.index.name = None
shar_t_table.columns.name = None


In [13]:
uniq_t_table

,Colon,Heart_Cell,Heart_Nuclei,Ileum,Kidney,Lung,Skin
Adipocytes,.,.,5136.0,.,.,.,.
Adventitial cells,.,.,.,.,.,4025.0,.
Alveolar adventitial fibroblast,.,.,.,.,.,5755.0,.
Atrial cardiomyocytes,.,.,38152.0,.,.,.,.
Basal cells,.,.,.,.,.,1660.0,.
Basophil,.,.,.,.,.,847.0,.
Bronchial SMCs,.,.,.,.,.,1141.0,.
Capillary endothelium,.,.,.,.,.,20340.0,.
Club cells,.,.,.,.,.,4868.0,.
Dendritic cells,.,.,.,.,.,.,4435.0


In [14]:
shar_t_table

,Colon,Heart_Cell,Heart_Nuclei,Ileum,Kidney,Lung,Skin
B cells,4303.0,.,.,7527.0,.,1007.0,.
CD4 T cells,7300.0,1483.0,2790.0,6810.0,1310.0,10474.0,11652.0
CD8 T cells,5311.0,1815.0,3674.0,7302.0,2962.0,12405.0,7263.0
Crypt stem cells,16277.0,.,.,5238.0,.,.,.
Endothelial cells,.,71050.0,41333.0,438.0,4147.0,.,.
Lymphatic endothelium,.,.,528.0,.,.,4052.0,4579.0
Enterocytes,16488.0,.,.,45157.0,.,.,.
Fibroblasts,6910.0,2778.0,118321.0,588.0,.,.,38917.0
Goblet cells,7801.0,.,.,9542.0,.,9815.0,.
Monocytes/macrophages,.,10523.0,29421.0,350.0,2343.0,154959.0,5503.0


In [15]:
ct_to_drop = [
    "Adipocytes",
    "Adventitial cells",
    "Alveolar adventitial fibroblast",
    "Basal cells",
    "Basophil",
    "Bronchial SMCs",
    "Club cells",
    "Melanocytes",
    "NKT",
    "Neural cells",
    "Neutrophil",
    "Proliferating_KC",
    "Tregs",
    "Tuft cell",
]

In [16]:
adata = adata[
    ~((adata.obs.cell_type.isin(ct_to_drop)) | (adata.obs.GSE_id == "GSE243444"))
]  # colon, two 10 yo donors]

In [17]:
# 1. Считаем количество клеток для каждой пары
counts_df = (
    adata.obs[["Tissue_global", "cell_type"]]
    .value_counts()
    .rename("count")
    .reset_index()
)

# 2. Разворачиваем в матричную таблицу
raw_pivot = counts_df.pivot(index="cell_type", columns="Tissue_global", values="count")
pivot_table = raw_pivot.fillna(".").astype(str)

pivot_table.index.name = None
pivot_table.columns.name = None

pivot_table

,Colon,Heart_Cell,Heart_Nuclei,Ileum,Kidney,Lung,Skin
Atrial cardiomyocytes,.,.,38152.0,.,.,.,.
B cells,3948.0,.,.,7527.0,.,1007.0,.
CD4 T cells,6250.0,1483.0,2790.0,6810.0,1310.0,10474.0,11652.0
CD8 T cells,5018.0,1815.0,3674.0,7302.0,2962.0,12405.0,7263.0
Capillary endothelium,.,.,.,.,.,20340.0,.
Crypt stem cells,15651.0,.,.,5238.0,.,.,.
Dendritic cells,.,.,.,.,.,.,4435.0
Diff. keratinocytes,.,.,.,.,.,.,36051.0
Endothelial cells,.,71050.0,41333.0,438.0,4147.0,.,.
Artery endothelium,.,.,.,.,.,17971.0,.


In [18]:
adata.obs.head()

,GSE_id,GSM_id,Age,Tissue_global,cell_type_final,Age_bin,cell_type
AAACCTGAGGCCGAAT-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,Fibroblast,21,Fibroblasts
AAACCTGAGTTAAGTG-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,CD8T,21,CD8 T cells
AAACCTGGTCTTCTCG-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,VE,21,Vein endothelium
AAACCTGGTGCAACTT-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,DC,21,Dendritic cells
AAACCTGTCAGCGACC-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,CD4T,21,CD4 T cells


In [35]:
n_expr_genes_per_cell2 = adata.X.getnnz(axis=1)

adata.obs["n_expr_genes"] = n_expr_genes_per_cell2

/tmp/ipykernel_84505/2560785777.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["n_expr_genes"] = n_expr_genes_per_cell2


In [37]:
adata.obs.to_csv(".././0_adata obs - reduced and filt.csv")

In [22]:
# Число ненулевых генов в каждой клетке
# if sparse.issparse(adata.X):
n_detected_genes = np.asarray((adata.X > 0).sum(axis=1)).ravel()

# Временный DataFrame
qc_df = adata.obs[["Tissue_global", "cell_type"]].copy()
qc_df["n_detected_genes"] = n_detected_genes

# Статистики для каждой пары Tissue × cell_type
gene_stats = (
    qc_df.groupby(["Tissue_global", "cell_type"], observed=True)["n_detected_genes"]
    .describe(percentiles=[0.25, 0.50, 0.75])
    .rename(
        columns={
            "min": "min",
            "25%": "q25",
            "mean": "mean",
            "50%": "median",
            "75%": "q75",
            "max": "max",
            "std": "std",
            "count": "n_cells",
        }
    )
)

gene_stats.astype("int")

n_cells  mean   std  min   q25  \
Tissue_global cell_type                                                        
Colon         B cells                            3948  1216   549  511   863   
              CD4 T cells                        6250  1225   546  503   796   
              CD8 T cells                        5018  1333   569  508   884   
              Crypt stem cells                  15651  3937  1667  525  2885   
              Enterocytes                       15321  2376  1404  502  1287   
              Fibroblasts                        4262  2296  1077  507  1467   
              Goblet cells                       7801  2648  1370  504  1630   
              Mast cells                          334  1351   624  555   876   
              Plasma cells                      20100  1658   753  502  1122   
              Mid-crypt cells                    1458  3940  1640  543  2889   
              γδ T cells                         1457  1773   575  530  1393   
Heart_Cell    CD4 T cells                        1483   934   415  500   637   
              CD8 T cells                        1815   960   406  501   680   
              Endothelial cells                 71050  1351   712  500   811   
              Fibroblasts                        2778  1623   720  500  1083   
              Monocytes/macrophages             10523  1307   589  500   840   
              Mural cells                       20566  1252   570  500   819   
              NK cells                           4121   971   384  500   707   
Heart_Nuclei  Atrial cardiomyocytes             38152  1889   965  500  1107   
              CD4 T cells                        2790  1044   409  500   732   
              CD8 T cells                        3674   964   374  500   675   
              Endothelial cells                 41333  1159   699  500   655   
              Lymphatic endothelium               528  1543   888  500   695   
              Fibroblasts                      118321  1430   860  500   750   
              Monocytes/macrophages             29421  1636   861  500   900   
              Mast cells                         1240   762   279  500   592   
              Mural cells                       61507   960   460  500   655   
              NK cells                           1621  1112   543  501   746   
              Ventricular cardiomyocytes       174929  1853   849  500  1190   
Ileum         B cells                            7527  1620   713  505  1108   
              CD4 T cells                        6810  1209   484  502   838   
              CD8 T cells                        7302  1391   577  503   966   
              Crypt stem cells                   5238  3075  1951  512  1517   
              Endothelial cells                   438  2583  1283  535  1567   
              Enterocytes                       45157  2643  1266  505  1766   
              Fibroblasts                         588  1974   831  530  1392   
              Goblet cells                       9542  2210  1243  504  1310   
              Monocytes/macrophages               350  2358  1116  538  1450   
              Paneth cells                       2407  3076  1558  531  1889   
              Plasma cells                       2838  1489   680  512   996   
              γδ T cells                          555  1554   493  516  1233   
Kidney        CD4 T cells                        1310   633   142  500   534   
              CD8 T cells                        2962   732   244  500   556   
              Endothelial cells                  4147  1411   727  500   829   
              Epithelial cells                  12205  1275   741  500   710   
              Monocytes/macrophages              2343   909   306  500   683   
              NK cells                           1432   757   305  500   565   
Lung          B cells                            1007  1633   817  562  1011   
              CD4 T cells                     

## Reveal sen gene sets

### cellage

In [41]:
cellage = pd.read_csv(
    ".././_gene_sets/cellage3.tsv",
    sep="\t",
)

In [42]:
cellage.head()

,Entrez ID,Gene symbol,Gene name,Cancer Cell,Type of senescence,Senescence Effect,Reference
0,22848,AAK1,AP2 associated kinase 1,No,Unclear,Induces,26583757
1,5243,ABCB1,ATP binding cassette subfamily B member 1,Yes,Stress-induced,Induces,10825123
2,368,ABCC6,ATP binding cassette subfamily C member 6,Yes,Unclear,Inhibits,28536638
3,51225,ABI3,ABI family member 3,Yes,Oncogene-induced,Induces,21223585
4,25890,ABI3BP,ABI family member 3 binding protein,Yes,Unclear,Induces,18559958


In [44]:
cellage = cellage[
    (cellage["Senescence Effect"] == "Induces") & (cellage["Cancer Cell"] == "No")
]

In [45]:
cellage.shape

(254, 7)

In [46]:
cellage["Type of senescence"].unique()

array(['Unclear', 'Stress-induced', 'Oncogene-induced', 'Replicative'],
      dtype=object)

In [48]:
genes = list(set(cellage["Gene symbol"]))

In [49]:
len(genes)

223

In [50]:
len(set(genes) & set(adata.var_names))

222

### fridman_sen_up

In [52]:
import decoupler as dc

fr = dc.pp.read_gmt(".././_gene_sets/FRIDMAN_SENESCENCE_UP.v2026.1.Hs.gmt")
fr.head(2)

,source,target
0,FRIDMAN_SENESCENCE_UP,ALDH1A3
1,FRIDMAN_SENESCENCE_UP,AOPEP


In [53]:
fr.shape

(77, 2)

In [54]:
len(set(fr.target) & set(adata.var_names))

77